In [ ]:
!pip install deepeval bert-score rouge-score
!pip install -q groq spacy python-dotenv textstat
!python -m spacy download en_core_web_sm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.8/787.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.6 MB/s eta 0:00:00
  Created

In [ ]:
import json
import random
import spacy
import os
from collections import Counter, defaultdict
from typing import List, Dict, Any
from google.colab import files
import re
import time
from groq import Groq

In [3]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
from getpass import getpass
class MultiKeyGroqModel:
    def __init__(self, model="llama-3.1-8b-instant"):
        self.model_name = model

        # Setup multiple API keys
        self.api_keys = self._setup_api_keys()
        self.current_key_index = 0
        self.key_failures = {i: 0 for i in range(len(self.api_keys))}

        # Initialize with first key
        self.client = Groq(api_key=self.api_keys[self.current_key_index])
        print(f" Loaded {len(self.api_keys)} API keys | Model: {model}")

    def _setup_api_keys(self):
        """Get API keys from user"""
        keys = []
        print("\nEnter Groq API Keys (press Enter to finish, min 1 required):")

        key1 = getpass("API Key 1: ")
        if not key1:
            raise ValueError("At least 1 API key required!")
        keys.append(key1)

        for i in range(2, 6):
            key = getpass(f"API Key {i} (optional, press Enter to skip): ")
            if key:
                keys.append(key)
            else:
                break

        return keys

    def _switch_api_key(self):
        """Switch to next API key"""
        old_index = self.current_key_index
        self.current_key_index = (self.current_key_index + 1) % len(self.api_keys)

        if self.current_key_index == old_index and len(self.api_keys) == 1:
            print("Only 1 key available, waiting 60s...")
            time.sleep(60)
            return

        self.client = Groq(api_key=self.api_keys[self.current_key_index])
        print(f"🔄 Switched to API Key #{self.current_key_index + 1}")

    def _clean_json(self, text: str):
        if not text:
            return None
        text = text.strip()
        text = re.sub(r"^``````$", "", text, flags=re.IGNORECASE|re.DOTALL).strip()
        text = re.sub(r"^``````$", "", text, flags=re.IGNORECASE|re.DOTALL).strip()
        if "Here is the JSON" in text:
            match = re.search(r"(\{.*\})", text, flags=re.DOTALL)
            if match:
                return match.group(1)
        return text

    def generate(self, prompt: str, temperature: float = 0.7, max_tokens: int = 512, json_mode: bool = False) -> str:
        if json_mode:
            enhanced_prompt = prompt + "\n\nIMPORTANT: RESPOND WITH VALID JSON ONLY. No explanations."
            system_msg = "You are a JSON generator. Return valid JSON only."
        else:
            enhanced_prompt = prompt
            system_msg = "You are a helpful business analyst."

        max_retries = 5
        base_delay = 2

        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": system_msg},
                        {"role": "user", "content": enhanced_prompt}
                    ],
                    response_format={"type": "json_object"} if json_mode else {"type": "text"},
                    temperature=temperature,
                    max_tokens=max_tokens
                )

                content = response.choices[0].message.content

                # Reset failure count on success
                self.key_failures[self.current_key_index] = 0

                if json_mode:
                    cleaned = self._clean_json(content)
                    try:
                        json.loads(cleaned)
                        return cleaned
                    except:
                        continue
                else:
                    return content

            except Exception as e:
                error_str = str(e)

                # Handle rate limit (429)
                if "429" in error_str or "rate_limit" in error_str.lower():
                    self.key_failures[self.current_key_index] += 1

                    print(f"API Key #{self.current_key_index + 1} rate limited (attempt {attempt+1}/{max_retries})")

                    # If key failed 3+ times, switch immediately
                    if self.key_failures[self.current_key_index] >= 3:
                        print(f"API Key #{self.current_key_index + 1} exhausted!")
                        self._switch_api_key()
                        continue

                    # Normal backoff
                    delay = (base_delay * (2 ** attempt)) + random.uniform(1, 3)
                    print(f"⏳ Sleeping {delay:.2f}s...")
                    time.sleep(delay)

                    # Switch key on last attempt if multiple keys available
                    if attempt == max_retries - 1 and len(self.api_keys) > 1:
                        print("Switching to next API key...")
                        self._switch_api_key()

                # Other errors
                else:
                    print(f"API Error: {error_str}")
                    time.sleep(2)

                continue

        # All retries failed
        if json_mode:
            return json.dumps({"query": "fallback_error", "error": "all_keys_exhausted"})
        else:
            return "INSUFFICIENT_DATA: All API keys exhausted."

# Initialize with multi-key support
groq_model = MultiKeyGroqModel()
llm_generate = groq_model.generate
print(" Multi-key model configured!\n")



🔑 Enter Groq API Keys (press Enter to finish, min 1 required):
API Key 1: ··········
API Key 2 (optional, press Enter to skip): ··········
✅ Loaded 1 API keys | Model: llama-3.1-8b-instant
✅ Multi-key model configured!



In [ ]:
# ============================================================================
# LOAD BOTH FILES
# ============================================================================

import json

FPS_PATH = "/content/final_curated_dataset.json"          # ← change path if needed
CLUSTERS_PATH = "/content/clusters_final_local.json" # ← change path if needed

print("📂 Loading File 1: FPS-Sampled Transcripts...")
with open(FPS_PATH, "r") as f:
    fps_transcripts = json.load(f)

print("📂 Loading File 2: Intent Clusters...")
with open(CLUSTERS_PATH, "r") as f:
    intent_clusters = json.load(f)

print(f"\n Loaded {len(fps_transcripts)} FPS transcripts")
print(f" Loaded intent clusters for domains: {list(intent_clusters.keys())}\n")


📂 Loading File 1: FPS-Sampled Transcripts...
📂 Loading File 2: Intent Clusters...

✅ Loaded 124 FPS transcripts
✅ Loaded intent clusters for domains: ['Hotel', 'Flight', 'Retail', 'Banking', 'Telecom', 'Insurance']



In [7]:
# ============================================================================
# CONFIGURATION
# ============================================================================

SUBCATEGORIES = [
    "Business Outcome Events",
    "Customer Frustration & Sentiment Drivers",
    "Agent Behavior & Response Quality",
    "Process & Workflow Breakdowns",
    "Information Gaps & Misalignment",
    "Policy/Compliance/Eligibility Friction",
    "Operational Delays & Inefficiencies",
    "Comparative Variation",
    "Anomalies & Outlier Patterns"
]

QUERY_CONFIG = {
    "single_transcript": 40,
    "cluster_aggregate": 30,
    "cross_cluster": 20,
    "cross_domain": 15,
    "temporal": 10,
    "causal_chain": 10,
    "counterfactual": 10,
    "unanswerable": 20,
    "task2_followups": 50,
}

FOLLOWUP_PROBABILITY = 0.6  # 60% of Task-1 queries get follow-ups
MAX_FOLLOWUP_DEPTH = 5      # Each can have 1-5 follow-up levels


In [8]:
# ============================================================================
# PROMPTS - YOUR ORIGINAL STYLE (BETTER QUALITY)
# ============================================================================

SINGLE_TRANSCRIPT_PROMPT = """
You are generating analytical queries for business analysts studying call transcripts.

TRANSCRIPT EXCERPT:
{transcript_excerpt}

Domain: {domain}
Intent/Reasoning Context: {intent}

Generate 1 analytical query that:
1. Is answerable using ONLY patterns/evidence from this transcript
2. Focuses on one of these reasoning types: causal, behavioral, temporal, comparative, or conditional
3. Is 12-40 words long
4. Fits this subcategory: {subcategory}
5. Has {difficulty} difficulty level:
   - easy: Single observable pattern
   - medium: Combines 2 dimensions OR adds quantitative/temporal element
   - hard: Multi-hop logic with conditional/causal chains

Return ONLY valid JSON (no markdown):
{{
  "query": "<the analytical question>",
  "subcategory": "{subcategory}",
  "reasoning_type": "<causal|temporal|comparative|conditional|behavioral>"
}}
"""

CLUSTER_AGGREGATE_PROMPT = """
You are generating analytical queries for business analysts studying patterns across multiple call transcripts.

Domain: {domain}
Intent/Reasoning Context: {intent}
Number of transcripts in cluster: {num_transcripts}

Sample patterns from transcripts:
{sample_patterns}

Generate 1 analytical query that:
1. Asks about PATTERNS across these transcripts (not a single call)
2. Is answerable using evidence from the provided patterns
3. Focuses on one of these reasoning types: causal, behavioral, temporal, comparative, or conditional
4. Is 15-50 words long
5. Fits this subcategory: {subcategory}
6. Has {difficulty} difficulty level:
   - easy: Single observable pattern across calls
   - medium: Combines 2 dimensions OR adds quantitative/temporal element
   - hard: Multi-hop logic with conditional/causal chains

Return ONLY valid JSON (no markdown):
{{
  "query": "<the analytical question>",
  "subcategory": "{subcategory}",
  "reasoning_type": "<causal|temporal|comparative|conditional|behavioral>"
}}
"""

CROSS_CLUSTER_PROMPT = """
You are generating analytical queries comparing patterns between two different intent clusters within the same domain.

Domain: {domain}
Cluster 1 Intent: {intent1} ({num1} transcripts)
Cluster 2 Intent: {intent2} ({num2} transcripts)

Sample patterns from Cluster 1:
{patterns1}

Sample patterns from Cluster 2:
{patterns2}

Generate 1 analytical query that:
1. Compares or contrasts patterns between these two intent clusters
2. Is answerable using evidence from both clusters
3. Reasoning type: comparative (always for cross-cluster analysis)
4. Is 15-50 words long
5. Fits this subcategory: {subcategory}
6. Has {difficulty} difficulty level

Return ONLY valid JSON (no markdown):
{{
  "query": "<the comparative question>",
  "subcategory": "{subcategory}",
  "reasoning_type": "comparative"
}}
"""

CROSS_DOMAIN_PROMPT = """
You are generating analytical queries comparing patterns across two different business domains.

Domain 1: {domain1}
Intent: {intent1} ({num1} transcripts)

Domain 2: {domain2}
Intent: {intent2} ({num2} transcripts)

Sample patterns from Domain 1:
{patterns1}

Sample patterns from Domain 2:
{patterns2}

Generate 1 analytical query that:
1. Compares or contrasts patterns between these two domains
2. Is answerable using evidence from both domains
3. Reasoning type: comparative (always for cross-domain analysis)
4. Is 15-50 words long
5. Fits this subcategory: {subcategory}
6. Difficulty: hard (cross-domain is always complex)

Return ONLY valid JSON (no markdown):
{{
  "query": "<the comparative question>",
  "subcategory": "{subcategory}",
  "reasoning_type": "comparative"
}}
"""

TEMPORAL_PROMPT = """
You are generating analytical queries about temporal patterns in call transcripts.

Domain: {domain}
Intent/Reasoning Context: {intent}
Number of transcripts: {num_transcripts}

Sample patterns from transcripts:
{sample_patterns}

Generate 1 analytical query that:
1. Focuses on TEMPORAL aspects (timing, sequence, duration, frequency, trends over time)
2. Is answerable using evidence from the provided patterns
3. Reasoning type: temporal (always for time-based analysis)
4. Is 15-50 words long
5. Fits this subcategory: {subcategory}
6. Difficulty: medium (temporal analysis requires moderate complexity)

Return ONLY valid JSON (no markdown):
{{
  "query": "<the temporal question>",
  "subcategory": "{subcategory}",
  "reasoning_type": "temporal"
}}
"""

CAUSAL_CHAIN_PROMPT = """
You are generating analytical queries requiring multi-hop causal reasoning.

Domain: {domain}
Intent/Reasoning Context: {intent}
Number of transcripts: {num_transcripts}

Sample patterns from transcripts:
{sample_patterns}

Generate 1 analytical query that:
1. Requires tracing cause → effect → outcome relationships (multi-step reasoning)
2. Is answerable using evidence from the provided patterns
3. Reasoning type: causal (always for causal-chain analysis)
4. Is 20-60 words long
5. Fits this subcategory: {subcategory}
6. Difficulty: hard (causal chains require complex multi-hop logic)

Return ONLY valid JSON (no markdown):
{{
  "query": "<the causal-chain question>",
  "subcategory": "{subcategory}",
  "reasoning_type": "causal"
}}
"""

COUNTERFACTUAL_PROMPT = """
You are generating counterfactual analytical queries (what-if scenarios).

Domain: {domain}
Intent/Reasoning Context: {intent}
Number of transcripts: {num_transcripts}

Sample patterns from transcripts:
{sample_patterns}

Generate 1 analytical query that:
1. Asks a hypothetical "what if" question based on observed patterns
2. Is answerable by reasoning from the provided evidence
3. Reasoning type: conditional (always for counterfactual analysis)
4. Is 15-50 words long
5. Fits this subcategory: {subcategory}
6. Difficulty: hard (counterfactuals require complex conditional reasoning)

Return ONLY valid JSON (no markdown):
{{
  "query": "<the counterfactual question>",
  "subcategory": "{subcategory}",
  "reasoning_type": "conditional"
}}
"""

UNANSWERABLE_PROMPT = """
You are generating queries that LOOK valid but are NOT answerable from call transcripts alone.

Domain: {domain}

Generate 1 query that:
1. Requires external data, future predictions, or information not present in call transcripts
2. Appears reasonable on the surface
3. Is 10-35 words long
4. Fits one of these subcategories

Return ONLY valid JSON (no markdown):
{{
  "query": "<the unanswerable query>",
  "unanswerable_reason": "<brief explanation why it cannot be answered>",
  "subcategory": "<one of the 9 subcategories>"
}}
"""

ANSWER_SINGLE_PROMPT = """
You are a business analyst answering analytical queries about call transcripts.

Query: {query}

Relevant Transcript:
{transcript_context}

Domain: {domain}
Subcategory: {subcategory}

Generate a detailed analytical answer that:
1. Directly addresses the query using ONLY evidence from the transcript above
2. Cites specific behavioral patterns, keywords, timing, or observations
3. Provides concrete examples from the transcript
4. Is 80-200 words
5. Does NOT hallucinate data not present in the transcript

If the query CANNOT be fully answered from this transcript, respond with:
"INSUFFICIENT_DATA: [explain what information is missing]"

Answer:
"""

ANSWER_CLUSTER_PROMPT = """
You are a business analyst analyzing patterns across {num_transcripts} call transcripts.

Query: {query}

Evidence from ALL {num_transcripts} transcripts in this cluster:
{cluster_evidence}

Domain: {domain}
Subcategory: {subcategory}

Generate a detailed analytical answer that:
1. Identifies patterns across ALL the calls shown
2. Uses evidence from the transcripts
3. Mentions frequency/commonality where applicable
4. Provides concrete examples
5. Is 150-300 words
6. Synthesizes insights, not just listing summaries

If insufficient data to answer: "INSUFFICIENT_DATA: [reason]"

Answer:
"""

FOLLOWUP_DECISION_PROMPT = """
You are analyzing whether a query-answer pair has potential for follow-up questions.

Query: {query}
Answer: {answer}

Analyze the FOLLOW-UP POTENTIAL:
- How many interesting aspects remain unexplored?
- What depth of questioning would maximize insight?
- Rate complexity: simple, moderate, complex, very complex

Return ONLY valid JSON (no markdown):
{{
  "should_followup": true/false,
  "recommended_depth": 0-5,
  "reasoning": "<brief explanation>"
}}
"""

FOLLOWUP_GENERATION_PROMPT = """
You are generating a follow-up analytical query for a multi-turn conversation.

Conversation so far:
{conversation_history}

Generate 1 follow-up query that:
1. BUILDS ON the previous answers
2. Explores a new angle or adds specificity
3. Adds conditions, comparisons, or temporal refinement where appropriate
4. Is answerable from the same source
5. Is 15-45 words
6. Fits this subcategory: {subcategory}

Follow-up types: clarification, conditional, comparative, temporal, causal

Return ONLY valid JSON (no markdown):
{{
  "query": "<follow-up question>",
  "followup_type": "<clarification|conditional|comparative|temporal|causal>",
  "subcategory": "{subcategory}",
  "reasoning_type": "<causal|temporal|comparative|conditional|behavioral>"
}}
"""

FOLLOWUP_ANSWER_PROMPT = """
You are answering follow-up questions in a multi-turn conversation.

Conversation History:
{conversation_history}

Current Question:
{current_query}

Relevant Context:
{context}

Generate a detailed answer to the current question that:
1. Builds on the previous answers in the conversation
2. Uses ONLY evidence from the context provided
3. Is 80-200 words

If insufficient data: "INSUFFICIENT_DATA: [explain why]"

Answer:
"""


In [ ]:
# ============================================================================
# SCOPE 1: SINGLE-TRANSCRIPT
# ============================================================================

def generate_single_transcript_queries(transcripts, num):
    print(f"\n{'='*70}")
    print(f"SCOPE 1: Generating {num} Single-Transcript Queries")
    print(f"{'='*70}")

    queries = []
    diff_dist = {"easy": int(num*0.3), "medium": int(num*0.5), "hard": int(num*0.2)}
    query_count = 0

    for diff, count in diff_dist.items():
        print(f"\n📝 Generating {count} {diff} queries...")

        for i in range(count):
            t = random.choice(transcripts)
            full_text = " ".join([turn.get('text','') for turn in t.get('conversation',[])])
            excerpt = " ".join(full_text.split()[:300])
            subcat = random.choice(SUBCATEGORIES)

            prompt = SINGLE_TRANSCRIPT_PROMPT.format(
                transcript_excerpt=excerpt,
                domain=t.get('domain','Unknown'),
                intent=t.get('intent','Unknown'),
                subcategory=subcat,
                difficulty=diff
            )

            try:
                resp = llm_generate(prompt, json_mode=True, max_tokens=200)
                data = json.loads(resp)

                queries.append({
                    "id": f"task1_{query_count+1}",
                    "query": data["query"],
                    "task": "task1",
                    "query_scope": "single_transcript",
                    "subcategory": data.get("subcategory", subcat),
                    "reasoning_type": data.get("reasoning_type","unknown"),
                    "target_difficulty": diff,
                    "domain": t.get('domain'),
                    "intent": t.get('intent'),
                    "source_transcript_ids": [t.get('transcript_id')],
                    "source_transcript_excerpt": excerpt[:500],
                    "is_answerable": True
                })
                query_count += 1

                if query_count % 10 == 0:
                    print(f"  ✓ Generated {query_count} queries...")
            except Exception as e:
                print(f"  Error: {e}")
                continue

    print(f"\n Generated {len(queries)} single-transcript queries")
    return queries


In [ ]:
# ============================================================================
# SCOPE 2: CLUSTER-AGGREGATE
# ============================================================================

def generate_cluster_aggregate_queries(clusters, num):
    print(f"\n{'='*70}")
    print(f"SCOPE 2: Generating {num} Cluster-Aggregate Queries")
    print(f"{'='*70}")

    queries = []
    diff_dist = {"easy": int(num*0.3), "medium": int(num*0.5), "hard": int(num*0.2)}
    query_count = 0

    for diff, count in diff_dist.items():
        print(f"\n📝 Generating {count} {diff} queries...")

        for i in range(count):
            domain = random.choice(list(clusters.keys()))
            cluster = random.choice(clusters[domain])
            trans = cluster.get('selected_transcripts',[])

            if len(trans) < 3:
                continue

            sample = random.sample(trans, min(5, len(trans)))
            patterns = "\n".join([f"- {t.get('summary','')[:200]}" for t in sample])
            subcat = random.choice(SUBCATEGORIES)

            prompt = CLUSTER_AGGREGATE_PROMPT.format(
                domain=domain,
                intent=cluster.get('intent'),
                num_transcripts=len(trans),
                sample_patterns=patterns,
                subcategory=subcat,
                difficulty=diff
            )

            try:
                resp = llm_generate(prompt, json_mode=True, max_tokens=250)
                data = json.loads(resp)

                all_ids = [t.get('transcript_id') for t in trans if t.get('transcript_id')]

                queries.append({
                    "id": f"task1_{100+query_count}",
                    "query": data["query"],
                    "task": "task1",
                    "query_scope": "cluster_aggregate",
                    "subcategory": data.get("subcategory", subcat),
                    "reasoning_type": data.get("reasoning_type","unknown"),
                    "target_difficulty": diff,
                    "domain": domain,
                    "intent": cluster.get('intent'),
                    "source_transcript_ids": all_ids,
                    "num_source_transcripts": len(trans),
                    "is_answerable": True
                })
                query_count += 1

                if query_count % 10 == 0:
                    print(f"  ✓ Generated {query_count} queries...")
            except Exception as e:
                print(f"  Error: {e}")
                continue

    print(f"\n Generated {len(queries)} cluster-aggregate queries")
    return queries

In [ ]:
# ============================================================================
# SCOPE 3: CROSS-CLUSTER
# ============================================================================

def generate_cross_cluster_queries(clusters, num):
    print(f"\n{'='*70}")
    print(f"SCOPE 3: Generating {num} Cross-Cluster Queries")
    print(f"{'='*70}")

    queries = []
    diff_dist = {"easy": int(num*0.3), "medium": int(num*0.5), "hard": int(num*0.2)}
    query_count = 0

    for diff, count in diff_dist.items():
        for i in range(count):
            domain = random.choice(list(clusters.keys()))
            if len(clusters[domain]) < 2:
                continue

            c1, c2 = random.sample(clusters[domain], 2)
            t1 = c1.get('selected_transcripts',[])
            t2 = c2.get('selected_transcripts',[])

            if not t1 or not t2:
                continue

            p1 = "\n".join([f"- {t.get('summary','')[:150]}" for t in t1[:3]])
            p2 = "\n".join([f"- {t.get('summary','')[:150]}" for t in t2[:3]])
            subcat = random.choice(SUBCATEGORIES)

            prompt = CROSS_CLUSTER_PROMPT.format(
                domain=domain,
                intent1=c1.get('intent'),
                num1=len(t1),
                intent2=c2.get('intent'),
                num2=len(t2),
                patterns1=p1,
                patterns2=p2,
                subcategory=subcat,
                difficulty=diff
            )

            try:
                resp = llm_generate(prompt, json_mode=True, max_tokens=250)
                data = json.loads(resp)

                all_ids = [t.get('transcript_id') for t in t1+t2 if t.get('transcript_id')]

                queries.append({
                    "id": f"task1_{200+query_count}",
                    "query": data["query"],
                    "task": "task1",
                    "query_scope": "cross_cluster",
                    "subcategory": data.get("subcategory", subcat),
                    "reasoning_type": "comparative",
                    "target_difficulty": diff,
                    "domain": domain,
                    "intents": [c1.get('intent'), c2.get('intent')],
                    "source_transcript_ids": all_ids,
                    "num_source_transcripts": len(all_ids),
                    "is_answerable": True
                })
                query_count += 1
            except Exception as e:
                print(f"  Error: {e}")
                continue

    print(f"\n Generated {len(queries)} cross-cluster queries")
    return queries

# ============================================================================
# SCOPE 4: CROSS-DOMAIN
# ============================================================================

def generate_cross_domain_queries(clusters, num):
    print(f"\n{'='*70}")
    print(f"SCOPE 4: Generating {num} Cross-Domain Queries")
    print(f"{'='*70}")

    queries = []
    domains = list(clusters.keys())

    if len(domains) < 2:
        print("Need 2+ domains")
        return queries

    for i in range(num):
        d1, d2 = random.sample(domains, 2)
        c1 = random.choice(clusters[d1])
        c2 = random.choice(clusters[d2])

        t1 = c1.get('selected_transcripts',[])
        t2 = c2.get('selected_transcripts',[])

        if not t1 or not t2:
            continue

        p1 = "\n".join([f"- {t.get('summary','')[:150]}" for t in t1[:3]])
        p2 = "\n".join([f"- {t.get('summary','')[:150]}" for t in t2[:3]])
        subcat = random.choice(SUBCATEGORIES)

        prompt = CROSS_DOMAIN_PROMPT.format(
            domain1=d1,
            intent1=c1.get('intent'),
            num1=len(t1),
            domain2=d2,
            intent2=c2.get('intent'),
            num2=len(t2),
            patterns1=p1,
            patterns2=p2,
            subcategory=subcat
        )

        try:
            resp = llm_generate(prompt, json_mode=True, max_tokens=250)
            data = json.loads(resp)

            all_ids = [t.get('transcript_id') for t in t1+t2 if t.get('transcript_id')]

            queries.append({
                "id": f"task1_{300+i}",
                "query": data["query"],
                "task": "task1",
                "query_scope": "cross_domain",
                "subcategory": data.get("subcategory", subcat),
                "reasoning_type": "comparative",
                "target_difficulty": "hard",
                "domains": [d1, d2],
                "intents": [c1.get('intent'), c2.get('intent')],
                "source_transcript_ids": all_ids,
                "num_source_transcripts": len(all_ids),
                "is_answerable": True
            })
        except Exception as e:
            print(f"  Error: {e}")
            continue

    print(f"\n Generated {len(queries)} cross-domain queries")
    return queries

# ============================================================================
# SCOPE 5: TEMPORAL
# ============================================================================

def generate_temporal_queries(clusters, num):
    print(f"\n{'='*70}")
    print(f"SCOPE 5: Generating {num} Temporal Queries")
    print(f"{'='*70}")

    queries = []

    for i in range(num):
        domain = random.choice(list(clusters.keys()))
        cluster = random.choice(clusters[domain])
        trans = cluster.get('selected_transcripts',[])

        if len(trans) < 5:
            continue

        sample = random.sample(trans, min(5, len(trans)))
        patterns = "\n".join([f"- {t.get('summary','')[:200]}" for t in sample])
        subcat = random.choice(SUBCATEGORIES)

        prompt = TEMPORAL_PROMPT.format(
            domain=domain,
            intent=cluster.get('intent'),
            num_transcripts=len(trans),
            sample_patterns=patterns,
            subcategory=subcat
        )

        try:
            resp = llm_generate(prompt, json_mode=True, max_tokens=250)
            data = json.loads(resp)

            queries.append({
                "id": f"task1_{400+i}",
                "query": data["query"],
                "task": "task1",
                "query_scope": "temporal",
                "subcategory": data.get("subcategory", subcat),
                "reasoning_type": "temporal",
                "target_difficulty": "medium",
                "domain": domain,
                "intent": cluster.get('intent'),
                "source_transcript_ids": [t.get('transcript_id') for t in trans if t.get('transcript_id')],
                "num_source_transcripts": len(trans),
                "is_answerable": True
            })
        except Exception as e:
            print(f"  Error: {e}")
            continue

    print(f"\n Generated {len(queries)} temporal queries")
    return queries

# ============================================================================
# SCOPE 6: CAUSAL-CHAIN
# ============================================================================

def generate_causal_chain_queries(clusters, num):
    print(f"\n{'='*70}")
    print(f"SCOPE 6: Generating {num} Causal-Chain Queries")
    print(f"{'='*70}")

    queries = []

    for i in range(num):
        domain = random.choice(list(clusters.keys()))
        cluster = random.choice(clusters[domain])
        trans = cluster.get('selected_transcripts',[])

        if len(trans) < 5:
            continue

        sample = random.sample(trans, min(5, len(trans)))
        patterns = "\n".join([f"- {t.get('summary','')[:200]}" for t in sample])
        subcat = random.choice(SUBCATEGORIES)

        prompt = CAUSAL_CHAIN_PROMPT.format(
            domain=domain,
            intent=cluster.get('intent'),
            num_transcripts=len(trans),
            sample_patterns=patterns,
            subcategory=subcat
        )

        try:
            resp = llm_generate(prompt, json_mode=True, max_tokens=300)
            data = json.loads(resp)

            queries.append({
                "id": f"task1_{500+i}",
                "query": data["query"],
                "task": "task1",
                "query_scope": "causal_chain",
                "subcategory": data.get("subcategory", subcat),
                "reasoning_type": "causal",
                "target_difficulty": "hard",
                "domain": domain,
                "intent": cluster.get('intent'),
                "source_transcript_ids": [t.get('transcript_id') for t in trans if t.get('transcript_id')],
                "num_source_transcripts": len(trans),
                "is_answerable": True
            })
        except Exception as e:
            print(f"  Error: {e}")
            continue

    print(f"\n Generated {len(queries)} causal-chain queries")
    return queries

# ============================================================================
# SCOPE 7: COUNTERFACTUAL
# ============================================================================

def generate_counterfactual_queries(clusters, num):
    print(f"\n{'='*70}")
    print(f"SCOPE 7: Generating {num} Counterfactual Queries")
    print(f"{'='*70}")

    queries = []

    for i in range(num):
        domain = random.choice(list(clusters.keys()))
        cluster = random.choice(clusters[domain])
        trans = cluster.get('selected_transcripts',[])

        if len(trans) < 5:
            continue

        sample = random.sample(trans, min(5, len(trans)))
        patterns = "\n".join([f"- {t.get('summary','')[:200]}" for t in sample])
        subcat = random.choice(SUBCATEGORIES)

        prompt = COUNTERFACTUAL_PROMPT.format(
            domain=domain,
            intent=cluster.get('intent'),
            num_transcripts=len(trans),
            sample_patterns=patterns,
            subcategory=subcat
        )

        try:
            resp = llm_generate(prompt, json_mode=True, max_tokens=250)
            data = json.loads(resp)

            queries.append({
                "id": f"task1_{600+i}",
                "query": data["query"],
                "task": "task1",
                "query_scope": "counterfactual",
                "subcategory": data.get("subcategory", subcat),
                "reasoning_type": "conditional",
                "target_difficulty": "hard",
                "domain": domain,
                "intent": cluster.get('intent'),
                "source_transcript_ids": [t.get('transcript_id') for t in trans if t.get('transcript_id')],
                "num_source_transcripts": len(trans),
                "is_answerable": True
            })
        except Exception as e:
            print(f"  Error: {e}")
            continue

    print(f"\n Generated {len(queries)} counterfactual queries")
    return queries

# ============================================================================
# UNANSWERABLE QUERIES
# ============================================================================

def generate_unanswerable_queries(num, domains):
    print(f"\n{'='*70}")
    print(f"Generating {num} Unanswerable Queries")
    print(f"{'='*70}")

    queries = []

    for i in range(num):
        domain = random.choice(domains)
        prompt = UNANSWERABLE_PROMPT.format(domain=domain)

        try:
            resp = llm_generate(prompt, json_mode=True, max_tokens=200, temperature=0.8)
            data = json.loads(resp)

            queries.append({
                "id": f"unanswerable_{i+1}",
                "query": data["query"],
                "task": "task1",
                "query_scope": "unanswerable",
                "subcategory": data.get("subcategory", random.choice(SUBCATEGORIES)),
                "unanswerable_reason": data.get("unanswerable_reason","unknown"),
                "domain": domain,
                "is_answerable": False
            })

            if (i+1) % 5 == 0:
                print(f"  ✓ Generated {i+1} queries...")
        except Exception as e:
            print(f"  Error: {e}")
            continue

    print(f"\n Generated {len(queries)} unanswerable queries")
    return queries

# ============================================================================
# ANSWER GENERATION
# ============================================================================

def generate_answers(queries, fps_transcripts, clusters):
    print(f"\n{'='*70}")
    print(f"Generating Answers for {len(queries)} Task-1 Queries")
    print(f"{'='*70}")

    transcript_lookup = {t.get('transcript_id'): t for t in fps_transcripts}

    answered_queries = []  # NEW: Only keep queries with valid answers
    failed_count = 0

    for i, q in enumerate(queries):
        scope = q.get('query_scope')

        # Skip unanswerable queries
        if scope == 'unanswerable':
            answered_queries.append(q)  # Keep unanswerable as-is
            continue

        # Single-transcript
        if scope == 'single_transcript':
            tid = q['source_transcript_ids'][0]
            t = transcript_lookup.get(tid)

            if t:
                full_text = " ".join([turn.get('text','') for turn in t.get('conversation',[])])
                context = " ".join(full_text.split()[:1000])

                prompt = ANSWER_SINGLE_PROMPT.format(
                    query=q['query'],
                    transcript_context=context,
                    domain=q.get('domain','Unknown'),
                    subcategory=q.get('subcategory','Unknown')
                )

                try:
                    ans = llm_generate(prompt, json_mode=False, max_tokens=400, temperature=0.3)

                    if "INSUFFICIENT_DATA" not in ans:
                        q['answer'] = {
                            'text': ans.strip(),
                            'method': 'single_transcript_analysis',
                            'source_transcript_ids': [tid],
                            'word_count': len(ans.split())
                        }
                        answered_queries.append(q)  # KEEP this query
                    else:
                        # REMOVE: Query couldn't be answered
                        print(f"  Removed {q['id']}: {ans[:50]}...")
                        failed_count += 1

                except Exception as e:
                    print(f"  Error on {q['id']}: {e}")
                    failed_count += 1
                    # Don't add to answered_queries (removed)

        # Multi-transcript scopes - USE ALL TRANSCRIPTS
        elif scope in ['cluster_aggregate', 'cross_cluster', 'cross_domain', 'temporal', 'causal_chain', 'counterfactual']:
            domain = q.get('domain') if 'domain' in q else q.get('domains',['Unknown'])[0]
            intent = q.get('intent')

            cluster = None
            if 'domain' in q and intent:
                cluster = next((c for c in clusters.get(domain,[]) if c.get('intent')==intent), None)

            if cluster:
                trans = cluster.get('selected_transcripts',[])

                # USE ALL TRANSCRIPTS (no sampling!)
                all_transcripts = trans

                # Build evidence from ALL summaries
                evidence = "\n\n".join([
                    f"Transcript {j+1} (ID: {t.get('transcript_id', 'unknown')[:8]}): {t.get('summary','')}"
                    for j, t in enumerate(all_transcripts)
                ])

                # Check if evidence is too large (optional safeguard)
                evidence_tokens = len(evidence.split()) * 1.3  # Rough token estimate

                if evidence_tokens > 25000:  # If exceeds ~25k tokens
                    print(f"  Warning: {q['id']} has {len(all_transcripts)} transcripts (~{int(evidence_tokens)} tokens)")
                    print(f"      This may exceed context window. Consider batching.")

                prompt = ANSWER_CLUSTER_PROMPT.format(
                    query=q['query'],
                    num_transcripts=len(all_transcripts),
                    cluster_evidence=evidence,
                    domain=domain,
                    subcategory=q.get('subcategory','Unknown')
                )

                try:
                    ans = llm_generate(prompt, json_mode=False, max_tokens=500, temperature=0.3)

                    if "INSUFFICIENT_DATA" not in ans:
                        q['answer'] = {
                            'text': ans.strip(),
                            'method': 'cluster_pattern_synthesis',
                            'source_transcript_ids': [t.get('transcript_id') for t in all_transcripts],
                            'num_transcripts_used': len(all_transcripts),
                            'word_count': len(ans.split())
                        }
                        answered_queries.append(q)  # KEEP this query
                    else:
                        print(f"  Removed {q['id']}: {ans[:50]}...")
                        failed_count += 1

                except Exception as e:
                    print(f"  Error on {q['id']}: {e}")
                    failed_count += 1
                    # Don't add to answered_queries

        if (i+1) % 10 == 0:
            print(f"  ✓ Processed {i+1}/{len(queries)}... (Kept: {len(answered_queries)}, Removed: {failed_count})")

    print(f"\n Successfully answered {len(answered_queries)} queries")
    print(f"❌ Removed {failed_count} unanswerable queries")

    return answered_queries

# ============================================================================
# TASK-2: MULTI-LEVEL FOLLOW-UPS
# ============================================================================

def generate_task2_followups(task1_with_answers, fps_transcripts, clusters, target_count):
    print(f"\n{'='*70}")
    print(f"TASK-2: Generating Multi-Level Follow-Ups")
    print(f"Target: ~{target_count} total Task-2 queries")
    print(f"{'='*70}")

    # Only use queries that have answers (already filtered)
    answerable = [q for q in task1_with_answers if q.get('answer')]
    print(f"\n Found {len(answerable)} answerable Task-1 queries")

    num_candidates = int(len(answerable) * FOLLOWUP_PROBABILITY)
    candidates = random.sample(answerable, min(num_candidates, len(answerable)))
    print(f" Selected {len(candidates)} candidates for follow-ups (60%)")

    transcript_lookup = {t.get('transcript_id'): t for t in fps_transcripts}

    all_task2 = []
    task2_count = 0
    removed_count = 0

    difficulty_options = ["easy", "medium", "hard"]
    difficulty_weights = [0.3, 0.5, 0.2]

    for base_q in candidates:
        if task2_count >= target_count:
            break

        # LLM decides depth
        decision_prompt = FOLLOWUP_DECISION_PROMPT.format(
            query=base_q['query'],
            answer=base_q['answer']['text'][:300]
        )

        try:
            dec_resp = llm_generate(decision_prompt, json_mode=True, temperature=0.7)
            dec_data = json.loads(dec_resp)

            if not dec_data.get('should_followup'):
                continue

            depth = min(dec_data.get('recommended_depth', 2), 4)
        except:
            depth = random.randint(1, 3)

        conv_history = f"Q1: {base_q['query']}\nA1: {base_q['answer']['text']}"
        parent_id = base_q['id']
        scope = base_q.get('query_scope')

        for level in range(1, depth + 1):
            if task2_count >= target_count:
                break

            # Rule-based difficulty
            if level == 1:
                followup_difficulty = random.choice(["easy", "medium"])
            elif level == 2:
                followup_difficulty = "medium"
            else:
                followup_difficulty = random.choice(["medium", "hard"])

            followup_prompt = FOLLOWUP_GENERATION_PROMPT.format(
                conversation_history=conv_history[-2000:],
                subcategory=base_q.get('subcategory', random.choice(SUBCATEGORIES)),
                difficulty=followup_difficulty
            )

            try:
                fu_resp = llm_generate(followup_prompt, json_mode=True, max_tokens=250, temperature=0.7)
                fu_data = json.loads(fu_resp)

                task2_q = {
                    "id": f"task2_{task2_count+1}_L{level}",
                    "query": fu_data['query'],
                    "task": "task2",
                    "query_scope": f"{scope}_followup",
                    "subcategory": fu_data.get('subcategory', base_q.get('subcategory')),
                    "reasoning_type": fu_data.get('reasoning_type', 'unknown'),
                    "followup_type": fu_data.get('followup_type', 'unknown'),
                    "target_difficulty": followup_difficulty,
                    "parent_query_id": parent_id,
                    "conversation_level": level + 1,
                    "followup_depth": depth,
                    "domain": base_q.get('domain'),
                    "intent": base_q.get('intent'),
                    "source_transcript_ids": base_q.get('source_transcript_ids'),
                    "is_answerable": True
                }

                # Generate answer
                if scope == 'single_transcript':
                    tid = base_q['source_transcript_ids'][0]
                    t = transcript_lookup.get(tid)

                    if t:
                        full_text = " ".join([turn.get('text','') for turn in t.get('conversation',[])])
                        context = " ".join(full_text.split()[:1000])

                        ans_prompt = FOLLOWUP_ANSWER_PROMPT.format(
                            conversation_history=conv_history[-1500:],
                            current_query=task2_q['query'],
                            context=context
                        )

                        try:
                            ans = llm_generate(ans_prompt, json_mode=False, max_tokens=400, temperature=0.3)

                            if "INSUFFICIENT_DATA" not in ans:
                                task2_q['answer'] = {
                                    'text': ans.strip(),
                                    'method': 'followup_single_transcript',
                                    'source_transcript_ids': [tid],
                                    'word_count': len(ans.split())
                                }
                                conv_history += f"\n\nQ{level+1}: {task2_q['query']}\nA{level+1}: {ans.strip()}"
                                parent_id = task2_q['id']
                                all_task2.append(task2_q)  # KEEP
                                task2_count += 1
                            else:
                                print(f"  Removed follow-up: {ans[:50]}...")
                                removed_count += 1
                                break  # Stop this chain
                        except:
                            removed_count += 1
                            break

                elif scope in ['cluster_aggregate', 'cross_cluster', 'cross_domain', 'temporal', 'causal_chain', 'counterfactual']:
                    domain = base_q.get('domain') if 'domain' in base_q else base_q.get('domains',['Unknown'])[0]
                    intent = base_q.get('intent')
                    cluster = next((c for c in clusters.get(domain,[]) if c.get('intent')==intent), None)

                    if cluster:
                        trans = cluster.get('selected_transcripts',[])

                        # USE ALL transcripts (same as parent)
                        all_transcripts = trans
                        evidence = "\n\n".join([f"Transcript: {t.get('summary','')}" for t in all_transcripts])

                        ans_prompt = FOLLOWUP_ANSWER_PROMPT.format(
                            conversation_history=conv_history[-1500:],
                            current_query=task2_q['query'],
                            context=evidence
                        )

                        try:
                            ans = llm_generate(ans_prompt, json_mode=False, max_tokens=400, temperature=0.3)

                            if "INSUFFICIENT_DATA" not in ans:
                                task2_q['answer'] = {
                                    'text': ans.strip(),
                                    'method': 'followup_cluster_synthesis',
                                    'source_transcript_ids': [t.get('transcript_id') for t in all_transcripts],
                                    'num_transcripts_used': len(all_transcripts),
                                    'word_count': len(ans.split())
                                }
                                conv_history += f"\n\nQ{level+1}: {task2_q['query']}\nA{level+1}: {ans.strip()}"
                                parent_id = task2_q['id']
                                all_task2.append(task2_q)  # KEEP
                                task2_count += 1
                            else:
                                print(f"  Removed follow-up: {ans[:50]}...")
                                removed_count += 1
                                break
                        except:
                            removed_count += 1
                            break

            except Exception as e:
                print(f"  Error: {e}")
                removed_count += 1
                break

        if task2_count % 5 == 0:
            print(f"  ✓ Generated {task2_count} Task-2 queries...")

    print(f"\n Generated {len(all_task2)} Task-2 queries")
    print(f"❌ Removed {removed_count} failed follow-ups")

    return all_task2

In [ ]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("\n" + "="*70)
print("🚀 ULTIMATE QUERY GENERATION PIPELINE")
print("="*70)

# Generate all Task-1 scopes
single = generate_single_transcript_queries(fps_transcripts, QUERY_CONFIG['single_transcript'])
cluster = generate_cluster_aggregate_queries(intent_clusters, QUERY_CONFIG['cluster_aggregate'])
cross_clust = generate_cross_cluster_queries(intent_clusters, QUERY_CONFIG['cross_cluster'])
cross_dom = generate_cross_domain_queries(intent_clusters, QUERY_CONFIG['cross_domain'])
temporal = generate_temporal_queries(intent_clusters, QUERY_CONFIG['temporal'])
causal = generate_causal_chain_queries(intent_clusters, QUERY_CONFIG['causal_chain'])
counterfact = generate_counterfactual_queries(intent_clusters, QUERY_CONFIG['counterfactual'])

domains = list(set([t.get('domain','Unknown') for t in fps_transcripts]))
unans = generate_unanswerable_queries(QUERY_CONFIG['unanswerable'], domains)

# Combine Task-1 (before filtering)
all_task1_generated = single + cluster + cross_clust + cross_dom + temporal + causal + counterfact
print(f"\n📊 Generated {len(all_task1_generated)} Task-1 queries (before filtering)")

# Generate answers - FILTERS OUT unanswerable queries
all_task1 = generate_answers(all_task1_generated, fps_transcripts, intent_clusters)
print(f"📊 After filtering: {len(all_task1)} valid Task-1 queries")

# Generate Task-2 follow-ups
task2 = generate_task2_followups(all_task1, fps_transcripts, intent_clusters, QUERY_CONFIG['task2_followups'])

# ============================================================================
# SAVE 2 FILES
# ============================================================================

print("\n💾 Saving files...")

# File 1: Task-1 Dataset (answerable + unanswerable)
task1_dataset = {
    "metadata": {
        "total_queries": len(all_task1) + len(unans),
        "answerable_queries": len(all_task1),
        "unanswerable_queries": len(unans),
        "queries_removed": len(all_task1_generated) - len(all_task1),
        "query_scope_distribution": dict(Counter(q.get('query_scope') for q in all_task1 + unans)),
        "subcategory_distribution": dict(Counter(q.get('subcategory') for q in all_task1 + unans)),
        "difficulty_distribution": dict(Counter(q.get('target_difficulty') for q in all_task1)),
        "scope_breakdown": {
            "single_transcript": len([q for q in all_task1 if q.get('query_scope') == 'single_transcript']),
            "cluster_aggregate": len([q for q in all_task1 if q.get('query_scope') == 'cluster_aggregate']),
            "cross_cluster": len([q for q in all_task1 if q.get('query_scope') == 'cross_cluster']),
            "cross_domain": len([q for q in all_task1 if q.get('query_scope') == 'cross_domain']),
            "temporal": len([q for q in all_task1 if q.get('query_scope') == 'temporal']),
            "causal_chain": len([q for q in all_task1 if q.get('query_scope') == 'causal_chain']),
            "counterfactual": len([q for q in all_task1 if q.get('query_scope') == 'counterfactual']),
            "unanswerable": len(unans)
        }
    },
    "answerable_queries": all_task1,
    "unanswerable_queries": unans
}

with open("task1_dataset.json", 'w') as f:
    json.dump(task1_dataset, f, indent=2)
print(" Saved: task1_dataset.json")

# File 2: Task-2 Dataset (follow-ups only)
task2_dataset = {
    "metadata": {
        "total_queries": len(task2),
        "query_scope_distribution": dict(Counter(q.get('query_scope') for q in task2)),
        "subcategory_distribution": dict(Counter(q.get('subcategory') for q in task2)),
        "difficulty_distribution": dict(Counter(q.get('target_difficulty') for q in task2)),
        "followup_depth_distribution": dict(Counter(q.get('followup_depth') for q in task2)),
        "conversation_level_distribution": dict(Counter(q.get('conversation_level') for q in task2)),
        "followup_type_distribution": dict(Counter(q.get('followup_type') for q in task2))
    },
    "queries": task2
}

with open("task2_dataset.json", 'w') as f:
    json.dump(task2_dataset, f, indent=2)
print(" Saved: task2_dataset.json")

# Download both files
print("\n📥 Downloading files...")
files.download("task1_dataset.json")
files.download("task2_dataset.json")

print("\n" + "="*70)
print("🎉 PIPELINE COMPLETE!")
print("="*70)
print(f"\nFiles Created:")
print(f"  1. task1_dataset.json  - {len(all_task1) + len(unans)} queries")
print(f"     • Answerable: {len(all_task1)}")
print(f"     • Unanswerable: {len(unans)}")
print(f"  2. task2_dataset.json  - {len(task2)} follow-up queries")
print(f"\n" + "="*70)
print(f"Task-1 Breakdown:")
print(f"="*70)
print(f"  Single-Transcript: {len(single)} generated")
print(f"  Cluster-Aggregate: {len(cluster)} generated")
print(f"  Cross-Cluster: {len(cross_clust)} generated")
print(f"  Cross-Domain: {len(cross_dom)} generated")
print(f"  Temporal: {len(temporal)} generated")
print(f"  Causal-Chain: {len(causal)} generated")
print(f"  Counterfactual: {len(counterfact)} generated")
print(f"  Unanswerable: {len(unans)} generated")
print(f"  ---")
print(f"  Total Generated: {len(all_task1_generated) + len(unans)}")
print(f"  Valid (after filtering): {len(all_task1)}")
print(f"  Removed: {len(all_task1_generated) - len(all_task1)}")
print(f"\n" + "="*70)
print(f"Task-2 Breakdown:")
print(f"="*70)
print(f"  Follow-ups Generated: {len(task2)}")
if task2:
    depth_dist = Counter([q.get('followup_depth') for q in task2])
    print(f"  Depth Distribution:")
    for depth, count in sorted(depth_dist.items()):
        print(f"    Depth {depth}: {count} chains")
print(f"\n" + "="*70)
print(f"TOTAL DATASET: {len(all_task1) + len(unans) + len(task2)} queries")
print(f"="*70)


In [ ]:
# Force download/save
from google.colab import files

if os.path.exists('task2_dataset.json'):
    files.download('task2_dataset.json')
else:
    print("File not found - check your pipeline's save location")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>